In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import re
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/3.global-data-cba/'
ground_data_name = 'ground_data_160226.csv'

In [ ]:
# derived paths
ground_data_path = os.path.join(base_dir, 'MRT-prediction/ground_data', ground_data_name)

#### fixing csv and georreferencing data

In [ ]:
df = pd.read_csv(ground_data_path, sep=";")
df

In [ ]:
# convert to numeric

cols = ["TG1", "TG2", "TA1", "TA2", "ANEMOMETRO"]


df[cols] = (
    df[cols]
    .astype(str)
    .apply(lambda x: x.str.replace(",", ".", regex=False))
    .apply(pd.to_numeric, errors="coerce")
)

# TG difference
df["dTG"] = df["TG1"] - df["TG2"]


In [ ]:
df

In [ ]:
# Scatter: TG1 vs dTG
plt.scatter(df["TG1"], df["dTG"])
plt.xlabel("TG1")
plt.ylabel("Diferencia en TG")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["TG1"]]   # predictor must be 2D
y = df["TG2"]

# Fit linear regression
model = LinearRegression()
model.fit(X, y)

# Coefficients
a = model.coef_[0]
b = model.intercept_

print(f"TG2 = {a:.3f} * TG1 + {b:.3f}")

# Apply correction: "pass" TG1 to TG2-equivalent
df["TG1_corr"] = a * df["TG1"] + b

reduction_rmse = (
    1 - np.sqrt((df["dTG_CORR"]**2).mean()) /
        np.sqrt((df["dTG"]**2).mean())
) * 100

print(f"RMSE reduction: {reduction_rmse:.1f} %")


In [ ]:
df["dTG_CORR"] = df["TG1_corr"] - df["TG2"]
# Scatter: TG1 vs dTG
plt.scatter(df["TG1_corr"], df["dTG_CORR"])
plt.xlabel("TG1_corr")
plt.ylabel("Dif TG adjusted")
plt.show()

In [ ]:
import numpy as np

metrics = pd.DataFrame({
    "metric": ["mean", "std", "RMSE", "MAE"],
    "dTG_raw": [
        df["dTG"].mean(),
        df["dTG"].std(),
        np.sqrt((df["dTG"]**2).mean()),
        df["dTG"].abs().mean()
    ],
    "dTG_corr": [
        df["dTG_CORR"].mean(),
        df["dTG_CORR"].std(),
        np.sqrt((df["dTG_CORR"]**2).mean()),
        df["dTG_CORR"].abs().mean()
    ]
})

print(metrics)


In [ ]:
reduction_rmse = (
    1 - np.sqrt((df["dTG_CORR"]**2).mean()) /
        np.sqrt((df["dTG"]**2).mean())
) * 100

print(f"RMSE reduction: {reduction_rmse:.1f} %")


In [ ]:
plt.boxplot(
    [df["dTG"].dropna(), df["dTG_CORR"].dropna()],
    labels=["dTG raw", "dTG corrected"]
)
plt.ylabel("TG1 − TG2")
plt.show()


MRT

In [ ]:
# Parameters
eps = 0.95
D1 = 0.04   # diameter of globe 1 [m] 
D2 = 0.05  # diameter of globe 2 [m] 

# Avoid zero wind (ISO recommendation)
v = df["ANEMOMETRO"].clip(lower=0.1)

# MRT from TG1
df["MRT1"] = (
    (
        (df["TG1"] + 273.15)**4
        + (1.1e8 * v**0.6 / (eps * D1**0.4)) * (df["TG1"] - df["TA1"])
    )**0.25
    - 273.15
)

# MRT from TG2
df["MRT2"] = (
    (
        (df["TG2"] + 273.15)**4
        + (1.1e8 * v**0.6 / (eps * D2**0.4)) * (df["TG2"] - df["TA2"])
    )**0.25
    - 273.15
)

# MRT difference
df["dMRT"] = df["MRT1"] - df["MRT2"]


In [ ]:
plt.scatter(
    df["TG1"],
    df["dMRT"],
    c=df["ANEMOMETRO"],
    cmap="viridis_r"
)

plt.xlabel("TG1")
plt.ylabel("ΔMRT")

cbar = plt.colorbar()
cbar.set_label("Wind speed (m/s)")

plt.show()


In [ ]:
# adjusting formula with my values

# MRT from TG1
df["MRT1"] = (
    (
        (df["TG1"] + 273.15)**4
        + (1.1e8 * v**0.6 / (eps * D1**0.4)) * (df["TG1"] - df["TA1"])
    )**0.25
    - 273.15
)

# MRT from TG2
df["MRT2"] = (
    (
        (df["TG2"] + 273.15)**4
        + (1.1e8 * v**1.5 / (eps * D2**0.4)) * (df["TG2"] - df["TA2"])
    )**0.25
    - 273.15
)

# MRT difference
df["dMRT"] = df["MRT1"] - df["MRT2"]



In [ ]:
plt.scatter(
    df["TG1"],
    df["dMRT"],
    c=df["ANEMOMETRO"],
    cmap="viridis_r"
)

plt.xlabel("TG1")
plt.ylabel("ΔMRT")

cbar = plt.colorbar()
cbar.set_label("Wind speed (m/s)")

plt.show()

not working so good. Better take points with no wind